In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from sklearn.manifold import MDS, TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# ─── PATHS ───────────────────────────────────────────────────────────────────
P_BASE = "/kaggle/input/datasets/d1ffic00lt/data-fusion-2026-case-3"
P_OUT  = "/kaggle/working/"
os.makedirs(P_OUT, exist_ok=True)

# ─── PALETTE ─────────────────────────────────────────────────────────────────
BG   = "#0d1117"
BG2  = "#161b22"
ACC1 = "#58a6ff"
ACC2 = "#3fb950"
ACC3 = "#f78166"
ACC4 = "#d2a8ff"
ACC5 = "#ffa657"
ACC6 = "#79c0ff"
GOLD = "#e3b341"
TEXT = "#e6edf3"
GRID = "#30363d"

PALETTE = [ACC1, ACC2, ACC3, ACC4, ACC5, GOLD, "#56d364", "#ff7b72", "#bc8cff"]
DAY_COLORS = {1:"#58a6ff",2:"#3fb950",3:"#ffa657",4:"#f78166",5:"#d2a8ff",6:"#79c0ff",7:"#e3b341"}

def style_ax(ax, title="", xlabel="", ylabel=""):
    ax.set_facecolor(BG2)
    ax.tick_params(colors=TEXT, labelsize=9)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID)
    ax.grid(True, color=GRID, linewidth=0.5, alpha=0.7)
    if title:  ax.set_title(title, color=TEXT, fontsize=11, fontweight='bold', pad=8)
    if xlabel: ax.set_xlabel(xlabel, color=TEXT, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT, fontsize=9)

def new_fig(w=16, h=9, title=""):
    fig = plt.figure(figsize=(w, h), facecolor=BG)
    if title:
        fig.suptitle(title, color=TEXT, fontsize=14, fontweight='bold', y=0.98)
    return fig

def save(fig, name):
    path = os.path.join(P_OUT, name)
    fig.savefig(path, dpi=140, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    print(f"  ✓ {name}")
    return path

# ─── LOAD DATA ───────────────────────────────────────────────────────────────
print("📂 Loading data...")
heroes  = pd.read_csv(os.path.join(P_BASE, "data_heroes.csv"))
objects = pd.read_csv(os.path.join(P_BASE, "data_objects.csv"))
ds_raw  = pd.read_csv(os.path.join(P_BASE, "dist_start.csv"))
do_raw  = pd.read_csv(os.path.join(P_BASE, "dist_objects.csv"), header=0)

dist_start  = dict(zip(ds_raw["object_id"], ds_raw["dist_start"]))
dist_matrix = do_raw.values.astype(np.float64)

N_H = len(heroes)
N_O = len(objects)
ND  = 7

print(f"  Heroes: {N_H}  |  Objects: {N_O}  |  Dist matrix: {dist_matrix.shape}")

# ─── DERIVED FEATURES ────────────────────────────────────────────────────────
objects["dist_from_start"] = objects["object_id"].map(dist_start)
objects["score_per_dist"]  = objects["reward"] / (objects["dist_from_start"] + 1)

for i, row in objects.iterrows():
    oid = int(row["object_id"])
    objects.at[i, "avg_dist_to_others"] = dist_matrix[oid-1].mean()
    objects.at[i, "min_dist_to_others"] = dist_matrix[oid-1][dist_matrix[oid-1]>0].min() if (dist_matrix[oid-1]>0).any() else 0

objects["reachability"] = objects["dist_from_start"] / objects["move_points_needed"] if "move_points_needed" in objects.columns else objects["dist_from_start"]
objects["cluster"] = KMeans(n_clusters=7, random_state=42, n_init=10).fit_predict(
    dist_matrix[:N_O, :N_O]
)

heroes["rank"] = heroes["move_points"].rank(ascending=False).astype(int)
heroes["tier"] = pd.cut(heroes["move_points"], bins=5, labels=["F","D","C","B","A"])

print("  ✓ Features computed\n")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — HERO STATS DASHBOARD
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 1: Hero Stats Dashboard")
fig = new_fig(18, 12, "⚔️  HERO STATISTICS DASHBOARD")
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.38)

# 1a — Move points distribution (histogram)
ax1 = fig.add_subplot(gs[0, :2])
vals = heroes["move_points"].values
ax1.hist(vals, bins=20, color=ACC1, edgecolor=BG, alpha=0.9)
ax1.axvline(vals.mean(), color=GOLD, lw=2, ls='--', label=f'Mean: {vals.mean():.0f}')
ax1.axvline(np.median(vals), color=ACC3, lw=2, ls=':', label=f'Median: {np.median(vals):.0f}')
style_ax(ax1, "Move Points Distribution", "Move Points", "Count")
ax1.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

# 1b — Sorted hero move points bar
ax2 = fig.add_subplot(gs[0, 2:])
sorted_h = heroes.sort_values("move_points", ascending=False)
colors_bar = [PALETTE[int(t) % len(PALETTE)] for t in sorted_h["tier"].cat.codes]
bars = ax2.bar(range(len(sorted_h)), sorted_h["move_points"].values, color=colors_bar, width=0.8)
style_ax(ax2, "All Heroes Sorted by Move Points", "Hero Rank", "Move Points")
ax2.set_xticks([])

# 1c — Box per tier
ax3 = fig.add_subplot(gs[1, :2])
tier_data = [heroes[heroes["tier"]==t]["move_points"].values for t in ["A","B","C","D","F"]]
bp = ax3.boxplot(tier_data, labels=["A","B","C","D","F"], patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], [ACC2, ACC1, ACC4, ACC5, ACC3]):
    patch.set_facecolor(color); patch.set_alpha(0.8)
for element in ['whiskers','caps','medians','fliers']:
    for item in bp[element]:
        item.set_color(TEXT)
style_ax(ax3, "Move Points by Tier", "Tier", "Move Points")

# 1d — CDF
ax4 = fig.add_subplot(gs[1, 2:])
sv = np.sort(vals)
cdf = np.arange(1, len(sv)+1) / len(sv)
ax4.plot(sv, cdf, color=ACC1, lw=2.5)
ax4.fill_between(sv, cdf, alpha=0.2, color=ACC1)
pct = [25, 50, 75, 90]
for p in pct:
    v = np.percentile(vals, p)
    ax4.axvline(v, color=GOLD, lw=1, ls='--', alpha=0.7)
    ax4.text(v, p/100, f'  p{p}', color=GOLD, fontsize=7, va='center')
style_ax(ax4, "CDF of Move Points", "Move Points", "Cumulative Probability")

# 1e — Scatter hero_id vs move_points colored by tier
ax5 = fig.add_subplot(gs[2, :2])
tier_map = {"A":ACC2,"B":ACC1,"C":ACC4,"D":ACC5,"F":ACC3}
for tier, grp in heroes.groupby("tier"):
    ax5.scatter(grp["hero_id"], grp["move_points"], c=tier_map.get(str(tier), TEXT),
                s=40, label=str(tier), alpha=0.85, edgecolors='none')
style_ax(ax5, "Hero ID vs Move Points (colored by tier)", "Hero ID", "Move Points")
ax5.legend(title="Tier", facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

# 1f — Pie of tiers
ax6 = fig.add_subplot(gs[2, 2:])
tc = heroes["tier"].value_counts().sort_index()
wedges, texts, autotexts = ax6.pie(tc.values, labels=tc.index, autopct='%1.0f%%',
    colors=[tier_map.get(str(t),TEXT) for t in tc.index],
    startangle=90, pctdistance=0.8,
    wedgeprops=dict(edgecolor=BG, linewidth=2))
for t in texts: t.set_color(TEXT); t.set_fontsize(10)
for at in autotexts: at.set_color(BG); at.set_fontsize(9); at.set_fontweight('bold')
ax6.set_facecolor(BG2)
ax6.set_title("Tier Distribution", color=TEXT, fontsize=11, fontweight='bold')

save(fig, "01_hero_dashboard.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — OBJECT STATS DASHBOARD
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 2: Object Stats Dashboard")
fig = new_fig(18, 14, "🏛️  OBJECT STATISTICS DASHBOARD")
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.38)

# 2a — Objects per day
ax = fig.add_subplot(gs[0, :2])
dc = objects["day_open"].value_counts().sort_index()
bars = ax.bar(dc.index, dc.values, color=[DAY_COLORS.get(d, ACC1) for d in dc.index], edgecolor=BG, width=0.7)
for bar, v in zip(bars, dc.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, str(v),
            ha='center', va='bottom', color=TEXT, fontsize=9, fontweight='bold')
style_ax(ax, "Objects Opened Per Day", "Day", "Count")
ax.set_xticks(range(1, ND+1))

# 2b — Cumulative rewards per day
ax = fig.add_subplot(gs[0, 2:])
cr = []
for d in range(1, ND+1):
    cr.append(objects[objects["day_open"] <= d]["reward"].sum())
ax.bar(range(1,ND+1), cr, color=GOLD, alpha=0.5, edgecolor=BG, width=0.7)
ax.plot(range(1,ND+1), cr, color=ACC3, lw=2.5, marker='o', markersize=7)
style_ax(ax, "Cumulative Reward Available by Day", "Day", "Total Reward")
ax.set_xticks(range(1, ND+1))

# 2c — Reward distribution
ax = fig.add_subplot(gs[1, :2])
rvals = objects["reward"].values
ax.hist(rvals, bins=30, color=ACC2, edgecolor=BG, alpha=0.9)
ax.axvline(rvals.mean(), color=GOLD, lw=2, ls='--', label=f'Mean: {rvals.mean():.0f}')
style_ax(ax, "Reward Distribution", "Reward", "Count")
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

# 2d — Distance from start distribution
ax = fig.add_subplot(gs[1, 2:])
dvals = objects["dist_from_start"].values
for d in range(1, ND+1):
    sub = objects[objects["day_open"]==d]["dist_from_start"].values
    if len(sub) > 0:
        ax.hist(sub, bins=20, alpha=0.6, color=DAY_COLORS[d], label=f"Day {d}", edgecolor='none')
style_ax(ax, "Distance from Start by Day Opened", "Distance", "Count")
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8, ncol=2)

# 2e — Score per distance ratio by day
ax = fig.add_subplot(gs[2, :2])
for d in range(1, ND+1):
    sub = objects[objects["day_open"]==d]
    ax.scatter(sub["dist_from_start"], sub["reward"],
               c=DAY_COLORS[d], s=30, alpha=0.7, label=f"Day {d}", edgecolors='none')
style_ax(ax, "Reward vs Distance from Start", "Distance from Start", "Reward")
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8, ncol=2)

# 2f — Avg dist to other objects per day (violin)
ax = fig.add_subplot(gs[2, 2:])
day_avgs = [objects[objects["day_open"]==d]["avg_dist_to_others"].values for d in range(1,ND+1)]
parts = ax.violinplot(day_avgs, positions=range(1,ND+1), showmedians=True, showextrema=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(list(DAY_COLORS.values())[i])
    pc.set_alpha(0.8)
parts['cmedians'].set_color(GOLD); parts['cmaxes'].set_color(TEXT); parts['cmins'].set_color(TEXT)
parts['cbars'].set_color(TEXT)
style_ax(ax, "Avg Distance to Others — by Day", "Day", "Avg Distance")

save(fig, "02_object_dashboard.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — DISTANCE MATRIX HEATMAPS
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 3: Distance Matrix Heatmaps")
fig = new_fig(18, 14, "📏  DISTANCE MATRIX ANALYSIS")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

dm = dist_matrix[:N_O, :N_O]

# 3a — Full heatmap (downsample to 100x100 for speed)
ax = fig.add_subplot(gs[0, :2])
step = max(1, N_O//100)
sub_dm = dm[::step, ::step]
cmap_heat = LinearSegmentedColormap.from_list("custom", [BG, ACC1, GOLD, ACC3])
im = ax.imshow(sub_dm, aspect='auto', cmap=cmap_heat, interpolation='nearest')
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02).ax.yaxis.set_tick_params(color=TEXT, labelcolor=TEXT)
ax.set_facecolor(BG2)
ax.set_title("Distance Matrix Heatmap (sampled)", color=TEXT, fontsize=11, fontweight='bold')
ax.tick_params(colors=TEXT, labelsize=8)

# 3b — Distance from start heatmap
ax = fig.add_subplot(gs[0, 2])
ds_arr = np.array([dist_start.get(i+1, 0) for i in range(N_O)])
side = int(np.ceil(np.sqrt(N_O)))
padded = np.pad(ds_arr, (0, side*side - N_O), constant_values=0).reshape(side, side)
im2 = ax.imshow(padded, cmap=cmap_heat, aspect='auto', interpolation='nearest')
plt.colorbar(im2, ax=ax, fraction=0.04, pad=0.02).ax.yaxis.set_tick_params(color=TEXT, labelcolor=TEXT)
ax.set_title("Distance from Start (grid view)", color=TEXT, fontsize=11, fontweight='bold')
ax.tick_params(colors=TEXT, labelsize=8)
ax.set_facecolor(BG2)

# 3c — Distribution of all pairwise distances
ax = fig.add_subplot(gs[1, :2])
flat = dm[np.triu_indices_from(dm, k=1)]
ax.hist(flat, bins=80, color=ACC4, edgecolor='none', alpha=0.85)
ax.axvline(flat.mean(), color=GOLD, lw=2, ls='--', label=f'Mean: {flat.mean():.0f}')
ax.axvline(np.percentile(flat, 90), color=ACC3, lw=1.5, ls=':', label=f'P90: {np.percentile(flat,90):.0f}')
style_ax(ax, "Distribution of All Pairwise Distances", "Distance", "Count")
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=9)

# 3d — Per-object: min neighbor distance
ax = fig.add_subplot(gs[1, 2])
np.fill_diagonal(dm, np.inf)
min_dists = dm.min(axis=1)
np.fill_diagonal(dm, 0)
ax.hist(min_dists, bins=30, color=ACC2, edgecolor='none', alpha=0.85)
style_ax(ax, "Min Distance to Nearest Neighbor", "Distance", "Count")
ax.axvline(min_dists.mean(), color=GOLD, lw=2, ls='--', label=f'Mean: {min_dists.mean():.0f}')
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=9)

save(fig, "03_distance_heatmaps.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — MDS / t-SNE SPATIAL EMBEDDING
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 4: MDS / t-SNE Spatial Embedding")

# MDS
print("   Computing MDS...")
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, n_init=1, max_iter=300)
coords_mds = mds.fit_transform(dm)

fig = new_fig(18, 14, "🗺️  SPATIAL EMBEDDINGS (MDS & t-SNE)")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 4a — MDS colored by day
ax = fig.add_subplot(gs[0, :2])
for d in range(1, ND+1):
    idx = objects[objects["day_open"]==d]["object_id"].values - 1
    idx = idx[idx < N_O]
    ax.scatter(coords_mds[idx, 0], coords_mds[idx, 1],
               c=DAY_COLORS[d], s=25, alpha=0.75, label=f"Day {d}", edgecolors='none')
# Mark start
ax.scatter([0], [0], c=GOLD, s=200, marker='*', zorder=5, label="Start")
style_ax(ax, "MDS Embedding — Colored by Day Opened", "MDS-1", "MDS-2")
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8, ncol=2)

# 4b — MDS colored by reward
ax = fig.add_subplot(gs[0, 2])
sc = ax.scatter(coords_mds[:, 0], coords_mds[:, 1],
                c=objects["reward"].values, cmap='plasma', s=18, alpha=0.75, edgecolors='none')
plt.colorbar(sc, ax=ax, fraction=0.04).ax.yaxis.set_tick_params(color=TEXT, labelcolor=TEXT)
style_ax(ax, "MDS — Colored by Reward", "MDS-1", "MDS-2")

# 4c — MDS colored by distance from start
ax = fig.add_subplot(gs[1, :2])
sc = ax.scatter(coords_mds[:, 0], coords_mds[:, 1],
                c=objects["dist_from_start"].values, cmap='coolwarm', s=18, alpha=0.75, edgecolors='none')
plt.colorbar(sc, ax=ax, fraction=0.02).ax.yaxis.set_tick_params(color=TEXT, labelcolor=TEXT)
style_ax(ax, "MDS — Colored by Distance from Start", "MDS-1", "MDS-2")

# 4d — MDS colored by cluster
ax = fig.add_subplot(gs[1, 2])
for cl in sorted(objects["cluster"].unique()):
    idx = objects[objects["cluster"]==cl]["object_id"].values - 1
    idx = idx[idx < N_O]
    ax.scatter(coords_mds[idx, 0], coords_mds[idx, 1],
               c=PALETTE[cl % len(PALETTE)], s=22, alpha=0.8, label=f"C{cl}", edgecolors='none')
style_ax(ax, "MDS — K-Means Clusters (k=7)", "MDS-1", "MDS-2")
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8, ncol=2)

save(fig, "04_spatial_embeddings.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 5 — REWARD & EFFICIENCY ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 5: Reward & Efficiency Analysis")
fig = new_fig(18, 12, "💰  REWARD & EFFICIENCY ANALYSIS")
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.38)

# 5a — Top-30 objects by reward
ax = fig.add_subplot(gs[0, :2])
top30 = objects.nlargest(30, "reward")
bars = ax.barh(range(30), top30["reward"].values[::-1],
               color=[DAY_COLORS[d] for d in top30["day_open"].values[::-1]], edgecolor='none')
ax.set_yticks(range(30))
ax.set_yticklabels([f"Obj {i}" for i in top30["object_id"].values[::-1]], fontsize=7, color=TEXT)
style_ax(ax, "Top 30 Objects by Reward", "Reward", "")

# 5b — Score/dist ratio top objects
ax = fig.add_subplot(gs[0, 2:])
top30e = objects.nlargest(30, "score_per_dist")
ax.barh(range(30), top30e["score_per_dist"].values[::-1],
        color=[DAY_COLORS[d] for d in top30e["day_open"].values[::-1]], edgecolor='none')
ax.set_yticks(range(30))
ax.set_yticklabels([f"Obj {i}" for i in top30e["object_id"].values[::-1]], fontsize=7, color=TEXT)
style_ax(ax, "Top 30 Objects by Reward/Distance Efficiency", "Efficiency (reward/dist)", "")

# 5c — Reward by day box
ax = fig.add_subplot(gs[1, :2])
data_by_day = [objects[objects["day_open"]==d]["reward"].values for d in range(1,ND+1)]
parts = ax.violinplot(data_by_day, positions=range(1,ND+1), showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(list(DAY_COLORS.values())[i]); pc.set_alpha(0.8)
parts['cmedians'].set_color(GOLD); parts['cbars'].set_color(TEXT)
parts['cmaxes'].set_color(TEXT); parts['cmins'].set_color(TEXT)
style_ax(ax, "Reward Distribution by Day (Violin)", "Day", "Reward")

# 5d — 2D hexbin: reward vs avg_dist
ax = fig.add_subplot(gs[1, 2:])
hb = ax.hexbin(objects["avg_dist_to_others"], objects["reward"],
               gridsize=25, cmap='plasma', mincnt=1)
plt.colorbar(hb, ax=ax, fraction=0.04).ax.yaxis.set_tick_params(color=TEXT, labelcolor=TEXT)
style_ax(ax, "Reward vs Avg Distance to Others", "Avg Distance", "Reward")

save(fig, "05_reward_efficiency.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 6 — HERO-OBJECT REACHABILITY
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 6: Hero-Object Reachability")
fig = new_fig(18, 12, "🎯  HERO-OBJECT REACHABILITY ANALYSIS")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# For each hero: how many day-1 objects can they reach on day 1?
VC = 100  # visit cost
reach_counts = []
for _, hero in heroes.iterrows():
    mp = hero["move_points"]
    day1_objects = objects[objects["day_open"]==1]["object_id"].values
    reachable = sum(1 for oid in day1_objects if dist_start.get(oid, 9999) + VC <= mp)
    reach_counts.append(reachable)
heroes["reachable_day1"] = reach_counts

# 6a — Hero reach count distribution
ax = fig.add_subplot(gs[0, :2])
ax.scatter(heroes["move_points"], heroes["reachable_day1"],
           c=ACC1, s=50, alpha=0.8, edgecolors=BG2)
ax.set_xlabel("Move Points", color=TEXT, fontsize=9)
ax.set_ylabel("Reachable Day-1 Objects", color=TEXT, fontsize=9)
style_ax(ax, "Hero Move Points vs Day-1 Reachable Objects", "Move Points", "Reachable Count")

# 6b — Reachability histogram
ax = fig.add_subplot(gs[0, 2])
ax.hist(heroes["reachable_day1"], bins=15, color=ACC2, edgecolor=BG)
style_ax(ax, "Distribution: Day-1 Reachable Objects", "Count Reachable", "Heroes")

# 6c — Objects reachable by % of heroes
ax = fig.add_subplot(gs[1, :2])
day1_obj = objects[objects["day_open"]==1].copy()
pcts = []
for _, obj in day1_obj.iterrows():
    d = dist_start.get(obj["object_id"], 9999)
    pcts.append(100 * (heroes["move_points"] >= d + VC).sum() / N_H)
day1_obj["pct_heroes"] = pcts
day1_obj_sorted = day1_obj.sort_values("pct_heroes", ascending=False)
ax.bar(range(len(day1_obj_sorted)), day1_obj_sorted["pct_heroes"].values,
       color=ACC4, edgecolor='none', width=0.8)
style_ax(ax, "Day-1 Objects — % of Heroes That Can Reach Them", "Object (sorted)", "% Heroes")
ax.axhline(50, color=GOLD, ls='--', lw=1.5, label='50%')
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

# 6d — Heatmap: hero mp vs object distance (reach matrix)
ax = fig.add_subplot(gs[1, 2])
hero_mps = np.sort(heroes["move_points"].unique())[:20]
obj_dists = np.sort(day1_obj["dist_from_start"].unique())[:20]
reach_mat = np.array([[1 if mp >= d + VC else 0 for d in obj_dists] for mp in hero_mps])
im = ax.imshow(reach_mat, aspect='auto', cmap=LinearSegmentedColormap.from_list("r", [ACC3, ACC2]))
ax.set_xlabel("Object (sorted by dist)", color=TEXT, fontsize=8)
ax.set_ylabel("Hero (sorted by MP)", color=TEXT, fontsize=8)
ax.tick_params(colors=TEXT, labelsize=7)
ax.set_title("Reach Matrix (top-20 sample)", color=TEXT, fontsize=10, fontweight='bold')
ax.set_facecolor(BG2)

save(fig, "06_reachability.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 7 — CLUSTER ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 7: Cluster Analysis")
fig = new_fig(18, 12, "🔬  CLUSTER ANALYSIS")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 7a — Cluster sizes
ax = fig.add_subplot(gs[0, 0])
cc = objects["cluster"].value_counts().sort_index()
ax.bar(cc.index, cc.values, color=[PALETTE[i%len(PALETTE)] for i in cc.index], edgecolor=BG)
style_ax(ax, "Cluster Sizes (k=7)", "Cluster", "Objects")

# 7b — Avg reward per cluster
ax = fig.add_subplot(gs[0, 1])
cr = objects.groupby("cluster")["reward"].mean()
ax.bar(cr.index, cr.values, color=[PALETTE[i%len(PALETTE)] for i in cr.index], edgecolor=BG)
ax.axhline(objects["reward"].mean(), color=GOLD, ls='--', lw=1.5, label='Overall mean')
style_ax(ax, "Avg Reward per Cluster", "Cluster", "Avg Reward")
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

# 7c — Avg dist from start per cluster
ax = fig.add_subplot(gs[0, 2])
cd = objects.groupby("cluster")["dist_from_start"].mean()
ax.bar(cd.index, cd.values, color=[PALETTE[i%len(PALETTE)] for i in cd.index], edgecolor=BG)
style_ax(ax, "Avg Distance from Start per Cluster", "Cluster", "Avg Dist")

# 7d — MDS with cluster hulls
ax = fig.add_subplot(gs[1, :2])
from matplotlib.patches import Ellipse
for cl in sorted(objects["cluster"].unique()):
    idx = objects[objects["cluster"]==cl]["object_id"].values - 1
    idx = idx[idx < len(coords_mds)]
    pts = coords_mds[idx]
    col = PALETTE[cl % len(PALETTE)]
    ax.scatter(pts[:, 0], pts[:, 1], c=col, s=20, alpha=0.7, label=f"C{cl}", edgecolors='none')
    if len(pts) >= 3:
        cx, cy = pts[:,0].mean(), pts[:,1].mean()
        sx, sy = pts[:,0].std()*2.2, pts[:,1].std()*2.2
        ell = Ellipse((cx, cy), sx*2, sy*2, alpha=0.12, color=col)
        ax.add_patch(ell)
style_ax(ax, "MDS + Cluster Ellipses", "MDS-1", "MDS-2")
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8, ncol=4)

# 7e — Cluster day composition stacked bar
ax = fig.add_subplot(gs[1, 2])
for cl in sorted(objects["cluster"].unique()):
    sub = objects[objects["cluster"]==cl]["day_open"].value_counts().sort_index()
    bottom = 0
    for d, cnt in sub.items():
        ax.bar(cl, cnt, bottom=bottom, color=DAY_COLORS.get(d, TEXT), edgecolor='none', width=0.7)
        bottom += cnt
style_ax(ax, "Day Composition per Cluster", "Cluster", "Object Count")
handles = [mpatches.Patch(color=DAY_COLORS[d], label=f"Day {d}") for d in range(1,ND+1)]
ax.legend(handles=handles, facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=7, ncol=2)

save(fig, "07_clusters.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 8 — ROUTE SIMULATION & SCORING ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 8: Scoring & Route Simulation")
fig = new_fig(18, 12, "📈  SCORING & ROUTE SIMULATION")
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.38)

# Simulate greedy routes for top heroes
def greedy_route(hero_mp, day, n_stops=5):
    """Simple greedy: pick closest high-reward objects for this day."""
    day_objs = objects[objects["day_open"]<=day].copy()
    day_objs["score_per_dist"] = day_objs["reward"] / (day_objs["dist_from_start"] + 1)
    top = day_objs.nlargest(min(n_stops*3, len(day_objs)), "score_per_dist")
    route_dist = 0
    visited = []
    pos = 0
    mp_left = hero_mp
    for _, obj in top.iterrows():
        d = dist_start.get(obj["object_id"], 9999) if pos == 0 else dm[pos-1, int(obj["object_id"])-1]
        if route_dist + d + VC <= hero_mp * len(visited) + hero_mp:
            visited.append(int(obj["object_id"]))
            route_dist += d + VC
            pos = int(obj["object_id"])
            if len(visited) >= n_stops: break
    return visited

# 8a — Hero move points vs estimated daily reward
ax = fig.add_subplot(gs[0, :2])
est_rewards = []
for _, hero in heroes.iterrows():
    mp = hero["move_points"]
    day1 = objects[(objects["day_open"]==1) & (objects["dist_from_start"] + VC <= mp)]
    est_rewards.append(day1["reward"].sum())
heroes["est_day1_reward"] = est_rewards
ax.scatter(heroes["move_points"], heroes["est_day1_reward"],
           c=heroes["move_points"], cmap='plasma', s=50, alpha=0.85, edgecolors='none')
style_ax(ax, "Hero MP vs Max Reachable Day-1 Reward", "Move Points", "Est. Reward")

# 8b — Score sensitivity: heroes needed vs penalty
ax = fig.add_subplot(gs[0, 2:])
hc_range = range(5, 26)
base_reward = 343000
scores = [base_reward - h * 2500 for h in hc_range]
ax.plot(list(hc_range), scores, color=ACC1, lw=2.5, marker='o', markersize=6)
ax.axhline(max(scores), color=GOLD, ls='--', lw=1.5, label=f'Max: {max(scores):,}')
ax.fill_between(list(hc_range), scores, min(scores), alpha=0.15, color=ACC1)
style_ax(ax, "Score vs Number of Heroes Used (same reward)", "Heroes Used", "Score")
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

# 8c — Reward per hero count 15-21
ax = fig.add_subplot(gs[1, :2])
hero_counts = range(15, 22)
best_scores = {
    15: 293500-5*2500, 16: 293500-4*2500, 17: 293500-3*2500,
    18: 293000-2*2500, 19: 293500-2500, 20: 293000, 21: 292500+2500
}
exp_scores = [base_reward - h*2500 for h in hero_counts]
ax.bar(list(hero_counts), exp_scores, color=ACC2, alpha=0.7, edgecolor=BG, width=0.7)
for x, y in zip(hero_counts, exp_scores):
    ax.text(x, y+500, f'{y:,}', ha='center', va='bottom', color=TEXT, fontsize=7)
style_ax(ax, "Theoretical Score by Heroes (base reward=343k)", "Heroes Used", "Score")

# 8d — Visit count needed vs reward
ax = fig.add_subplot(gs[1, 2:])
visits_range = range(600, 720, 5)
visit_scores = [v * 500 - 20*2500 for v in visits_range]
ax.plot(list(visits_range), visit_scores, color=ACC5, lw=2.5)
ax.axhline(342500, color=ACC3, ls='--', lw=1.5, label='Current best: 342,500')
ax.axhline(294500, color=GOLD, ls='--', lw=1.5, label='Target: 294,500')
style_ax(ax, "Score vs Visits (20 heroes, 500 reward/visit)", "Visits", "Score")
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

save(fig, "08_scoring.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 9 — OBJECT NETWORK GRAPH (proximity)
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 9: Object Proximity Network")
import networkx as nx

fig = new_fig(18, 12, "🕸️  OBJECT PROXIMITY NETWORK")
gs  = gridspec.GridSpec(1, 2, figure=fig, hspace=0.3, wspace=0.3)

# Build sparse graph: connect objects closer than threshold
THRESHOLD = 200
sample_ids = list(range(0, min(100, N_O)))  # first 100 objects for clarity

G = nx.Graph()
for i in sample_ids:
    G.add_node(i+1,
               day=int(objects.loc[i, "day_open"]),
               reward=int(objects.loc[i, "reward"]),
               dist_start=int(objects.loc[i, "dist_from_start"]))

edges_added = 0
for ii in range(len(sample_ids)):
    for jj in range(ii+1, len(sample_ids)):
        i, j = sample_ids[ii], sample_ids[jj]
        if dm[i, j] < THRESHOLD and dm[i, j] > 0:
            G.add_edge(i+1, j+1, weight=float(dm[i, j]))
            edges_added += 1

print(f"   Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

pos = {n: (coords_mds[n-1, 0], coords_mds[n-1, 1]) for n in G.nodes()}

ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(BG2)
node_colors = [DAY_COLORS[G.nodes[n]['day']] for n in G.nodes()]
node_sizes  = [G.nodes[n]['reward'] / 25 for n in G.nodes()]
edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
max_w = max(edge_weights) if edge_weights else 1
edge_alphas = [0.1 + 0.5*(1 - w/max_w) for w in edge_weights]
nx.draw_networkx_edges(G, pos, ax=ax1, alpha=0.2, edge_color=GRID, width=0.5)
nx.draw_networkx_nodes(G, pos, ax=ax1, node_color=node_colors, node_size=node_sizes, alpha=0.9)
top_nodes = sorted(G.nodes(), key=lambda n: G.nodes[n]['reward'], reverse=True)[:10]
nx.draw_networkx_labels(G, pos, labels={n: str(n) for n in top_nodes}, ax=ax1,
                        font_size=7, font_color=TEXT)
ax1.set_title(f"Proximity Network (d<{THRESHOLD}), colored by day, size=reward",
              color=TEXT, fontsize=10, fontweight='bold')
handles2 = [mpatches.Patch(color=DAY_COLORS[d], label=f"Day {d}") for d in range(1, ND+1)]
ax1.legend(handles=handles2, facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8, loc='upper left')
for spine in ax1.spines.values(): spine.set_edgecolor(GRID)
ax1.tick_params(colors=TEXT)

# 9b — Degree distribution
ax2 = fig.add_subplot(gs[0, 1])
degrees = [d for _, d in G.degree()]
ax2.hist(degrees, bins=max(degrees)+1 if degrees else 10, color=ACC4, edgecolor=BG)
style_ax(ax2, f"Node Degree Distribution (threshold={THRESHOLD})", "Degree", "Count")
if degrees:
    ax2.axvline(np.mean(degrees), color=GOLD, ls='--', lw=2, label=f'Mean: {np.mean(degrees):.1f}')
    ax2.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

save(fig, "09_network_graph.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 10 — HIERARCHICAL CLUSTERING DENDROGRAM
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 10: Dendrogram")
fig = new_fig(18, 10, "🌳  HIERARCHICAL CLUSTERING DENDROGRAM (50 objects sample)")

# Use 50 objects for readability
samp = list(range(0, 50))
samp_dm = dm[np.ix_(samp, samp)]
condensed = squareform(samp_dm)
Z = linkage(condensed, method='ward')

ax = fig.add_subplot(111)
ax.set_facecolor(BG2)
dn = dendrogram(Z, ax=ax, color_threshold=Z[-10, 2],
                above_threshold_color=TEXT,
                leaf_font_size=7,
                labels=[f"O{i+1}" for i in samp])
ax.tick_params(colors=TEXT, labelsize=7)
ax.set_title("Ward Linkage Dendrogram — Objects 1-50", color=TEXT, fontsize=12, fontweight='bold')
ax.set_xlabel("Object", color=TEXT, fontsize=9)
ax.set_ylabel("Distance", color=TEXT, fontsize=9)
for spine in ax.spines.values(): spine.set_edgecolor(GRID)
ax.grid(True, color=GRID, linewidth=0.5, alpha=0.5, axis='y')

save(fig, "10_dendrogram.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 11 — DAY-BY-DAY PROGRESSION
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 11: Day-by-Day Progression")
fig = new_fig(18, 10, "📅  DAY-BY-DAY GAME PROGRESSION")
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.38)

# 11a — Cumulative objects opened
ax = fig.add_subplot(gs[0, :2])
cum = []
for d in range(1, ND+1):
    cum.append(len(objects[objects["day_open"] <= d]))
ax.step(range(1, ND+1), cum, where='post', color=ACC1, lw=2.5)
ax.fill_between(range(1, ND+1), cum, step='post', alpha=0.2, color=ACC1)
for d, c in zip(range(1, ND+1), cum):
    ax.text(d, c+5, str(c), ha='center', color=TEXT, fontsize=9)
style_ax(ax, "Cumulative Objects Available", "Day", "Total Objects")
ax.set_xticks(range(1, ND+1))

# 11b — New objects per day
ax = fig.add_subplot(gs[0, 2])
new_per_day = [len(objects[objects["day_open"]==d]) for d in range(1, ND+1)]
ax.bar(range(1, ND+1), new_per_day, color=list(DAY_COLORS.values()), edgecolor=BG)
style_ax(ax, "New Objects Per Day", "Day", "New Objects")
ax.set_xticks(range(1, ND+1))

# 11c — Reward density per day
ax = fig.add_subplot(gs[0, 3])
rew_per_day = [objects[objects["day_open"]==d]["reward"].sum() for d in range(1, ND+1)]
ax.bar(range(1, ND+1), rew_per_day, color=list(DAY_COLORS.values()), edgecolor=BG)
style_ax(ax, "Total Reward Per Day", "Day", "Total Reward")
ax.set_xticks(range(1, ND+1))

# 11d — Day accessibility: fraction of all heroes that can visit avg object that day
ax = fig.add_subplot(gs[1, :2])
accessibility = []
for d in range(1, ND+1):
    day_o = objects[objects["day_open"]==d]
    if len(day_o) == 0:
        accessibility.append(0)
        continue
    avg_d = day_o["dist_from_start"].mean()
    frac = (heroes["move_points"] >= avg_d + VC).mean()
    accessibility.append(frac * 100)
ax.bar(range(1, ND+1), accessibility, color=list(DAY_COLORS.values()), edgecolor=BG)
style_ax(ax, "% Heroes Able to Visit Avg Object (by day)", "Day", "% Heroes")
ax.set_xticks(range(1, ND+1))

# 11e — Competition (reward per available hero capacity)
ax = fig.add_subplot(gs[1, 2:])
eff = [r/max(a, 0.01) for r, a in zip(rew_per_day, accessibility)]
ax.plot(range(1, ND+1), eff, color=ACC5, lw=2.5, marker='D', markersize=8)
ax.fill_between(range(1, ND+1), eff, alpha=0.2, color=ACC5)
style_ax(ax, "Reward Density / Hero Accessibility", "Day", "Efficiency")
ax.set_xticks(range(1, ND+1))

save(fig, "11_day_progression.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 12 — CORRELATION & FEATURE MATRIX
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 12: Correlation Matrix")
fig = new_fig(14, 12, "🔗  FEATURE CORRELATION MATRIX")

feats = objects[["object_id","day_open","reward","dist_from_start",
                  "avg_dist_to_others","min_dist_to_others","score_per_dist","cluster"]].copy()
corr = feats.corr()

ax = fig.add_subplot(111)
cmap_corr = LinearSegmentedColormap.from_list("corr", [ACC3, BG2, ACC2])
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
im = ax.imshow(corr.values, cmap=cmap_corr, vmin=-1, vmax=1, aspect='auto')

for i in range(len(corr)):
    for j in range(len(corr)):
        val = corr.iloc[i, j]
        color = TEXT if abs(val) < 0.6 else BG
        ax.text(j, i, f"{val:.2f}", ha='center', va='center', color=color, fontsize=9, fontweight='bold')

ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', color=TEXT, fontsize=9)
ax.set_yticklabels(corr.columns, color=TEXT, fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.03).ax.yaxis.set_tick_params(color=TEXT, labelcolor=TEXT)
ax.set_facecolor(BG2)
ax.set_title("Feature Correlation Matrix", color=TEXT, fontsize=13, fontweight='bold')
for spine in ax.spines.values(): spine.set_edgecolor(GRID)

save(fig, "12_correlations.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 13 — OPTIMAL HERO COUNT ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 13: Optimal Hero Count Analysis")
fig = new_fig(18, 10, "🏆  OPTIMAL HERO COUNT & SCORE BREAKDOWN")
gs  = gridspec.GridSpec(1, 3, figure=fig, hspace=0.3, wspace=0.4)

# Known experiment results from user's log
exp_data = [
    {"heroes": 20, "t1": 120, "tn": 10, "reward": 342500, "visits": 685, "score": 292500},
    {"heroes": 20, "t1": 180, "tn": 15, "reward": 343000, "visits": 686, "score": 293000},
    {"heroes": 19, "t1": 120, "tn": 10, "reward": None, "visits": None, "score": None},
    {"heroes": 19, "t1": 180, "tn": 15, "reward": None, "visits": None, "score": None},
    {"heroes": 21, "t1": 120, "tn": 10, "reward": None, "visits": None, "score": None},
    {"heroes": 21, "t1": 180, "tn": 15, "reward": None, "visits": None, "score": None},
    {"heroes": 18, "t1": 180, "tn": 15, "reward": None, "visits": None, "score": None},
]
df_exp = pd.DataFrame(exp_data)
df_known = df_exp.dropna()

# 13a — Score by hero count (all combos theoretical)
ax = fig.add_subplot(gs[0])
hero_range = range(15, 26)
for base_reward in [340000, 342500, 343000, 345000, 350000]:
    scores_hr = [base_reward - h*2500 for h in hero_range]
    lbl = f"Reward={base_reward//1000}k"
    col = GOLD if base_reward == 343000 else (ACC2 if base_reward == 350000 else GRID)
    lw  = 2.5 if base_reward in [343000, 350000] else 1
    ax.plot(list(hero_range), scores_hr, color=col, lw=lw, label=lbl, marker='o' if lw>1 else None, markersize=5)
ax.axhline(294500, color=ACC3, ls='--', lw=2, label='Target 294,500')
ax.axvline(20, color=ACC4, ls=':', lw=1.5, label='n=20')
style_ax(ax, "Score by Hero Count\n(varying base reward)", "Heroes Used", "Score")
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=7)

# 13b — Score breakdown waterfall for known experiments
ax = fig.add_subplot(gs[1])
if len(df_known) > 0:
    x = range(len(df_known))
    ax.bar(x, df_known["reward"], color=ACC2, alpha=0.8, label="Reward", edgecolor=BG)
    ax.bar(x, -df_known["heroes"]*2500, color=ACC3, alpha=0.8, label="Hero Cost", edgecolor=BG)
    ax.bar(x, df_known["score"], color=ACC1, alpha=0.5, label="Score", edgecolor=BG, width=0.4)
    ax.set_xticks(list(x))
    ax.set_xticklabels([f"E{i+1}\nH={int(r['heroes'])}" for i, r in df_known.iterrows()], color=TEXT, fontsize=8)
style_ax(ax, "Known Experiment Breakdown", "Experiment", "Value")
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

# 13c — How much more reward needed to beat 294500 with 20 heroes
ax = fig.add_subplot(gs[2])
current_best = 293000
target = 294500
gap = target - current_best
reward_needed = 343000 + gap  # same hero cost
x_labels = ["Current\nBest", "Gap\nNeeded", "Target\nReward", "Target\nScore"]
vals_bar  = [293000, gap, reward_needed, 294500]
colors_b  = [ACC1, ACC5, ACC2, GOLD]
bars = ax.bar(x_labels, vals_bar, color=colors_b, edgecolor=BG, width=0.6)
for bar, val in zip(bars, vals_bar):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
            f'{val:,}', ha='center', color=TEXT, fontsize=8, fontweight='bold')
style_ax(ax, f"Gap to Target 294,500\n(+{gap:,} score needed)", "", "Score")
ax.set_ylim(0, max(vals_bar)*1.12)

save(fig, "13_optimal_heroes.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 14 — SUMMARY TABLE (saved as PNG)
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Fig 14: Summary Table")
fig = new_fig(18, 8, "📋  DATA SUMMARY STATISTICS")
gs  = gridspec.GridSpec(1, 3, figure=fig, hspace=0.3, wspace=0.3)

def make_table(ax, data, col_labels, row_labels, title):
    ax.set_facecolor(BG2)
    ax.axis('off')
    ax.set_title(title, color=TEXT, fontsize=11, fontweight='bold', pad=8)
    t = ax.table(cellText=data, colLabels=col_labels, rowLabels=row_labels,
                 loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(9)
    for (row, col), cell in t.get_celld().items():
        cell.set_facecolor(BG if row == 0 else BG2)
        cell.set_edgecolor(GRID)
        cell.set_text_props(color=TEXT if row > 0 else GOLD)
    return t

# Table 1: Heroes summary
ax = fig.add_subplot(gs[0])
h_stats = [
    [N_H, f"{heroes['move_points'].mean():.0f}", f"{heroes['move_points'].min()}", f"{heroes['move_points'].max()}"]
]
make_table(ax, h_stats, ["Count","Avg MP","Min MP","Max MP"], ["Heroes"], "Hero Summary")

# Table 2: Objects per day
ax = fig.add_subplot(gs[1])
obj_data = [[d, len(objects[objects["day_open"]==d]),
             f"{objects[objects['day_open']==d]['reward'].mean():.0f}",
             f"{objects[objects['day_open']==d]['dist_from_start'].mean():.0f}"]
            for d in range(1, ND+1)]
make_table(ax, obj_data, ["Day","Count","Avg Reward","Avg Dist"], list(range(1,ND+1)), "Objects per Day")

# Table 3: Distance stats
ax = fig.add_subplot(gs[2])
ds_vals = list(dist_start.values())
flat_d  = dm[np.triu_indices_from(dm, k=1)]
dist_data = [
    [f"{np.mean(ds_vals):.0f}", f"{np.min(ds_vals)}", f"{np.max(ds_vals)}"],
    [f"{flat_d.mean():.0f}",    f"{flat_d.min():.0f}", f"{flat_d.max():.0f}"],
]
make_table(ax, dist_data, ["Mean","Min","Max"], ["Dist-Start","Obj-Obj"], "Distance Stats")

save(fig, "14_summary_table.png")

print("\n✅  All 14 figures saved!")
print(f"📁  Output: {P_OUT}")

# Summary stats printout
print("\n" + "="*60)
print("📊 DATA SUMMARY")
print("="*60)
print(f"Heroes:  {N_H}")
print(f"  MP range: {heroes['move_points'].min()} – {heroes['move_points'].max()}")
print(f"  MP mean:  {heroes['move_points'].mean():.0f}")
print(f"\nObjects: {N_O}")
print(f"  Reward range:   {objects['reward'].min()} – {objects['reward'].max()}")
print(f"  Total reward:   {objects['reward'].sum():,}")
print(f"  Avg dist start: {objects['dist_from_start'].mean():.0f}")
print(f"\nDistance matrix: {dist_matrix.shape}")
print(f"  Min pairwise: {flat_d.min():.0f}")
print(f"  Max pairwise: {flat_d.max():.0f}")
print(f"  Mean pairwise:{flat_d.mean():.0f}")
print("="*60)

📂 Loading data...
  Heroes: 100  |  Objects: 700  |  Dist matrix: (700, 700)
  ✓ Features computed

📊 Fig 1: Hero Stats Dashboard
  ✓ 01_hero_dashboard.png
📊 Fig 2: Object Stats Dashboard
  ✓ 02_object_dashboard.png
📊 Fig 3: Distance Matrix Heatmaps
  ✓ 03_distance_heatmaps.png
📊 Fig 4: MDS / t-SNE Spatial Embedding
   Computing MDS...
  ✓ 04_spatial_embeddings.png
📊 Fig 5: Reward & Efficiency Analysis
  ✓ 05_reward_efficiency.png
📊 Fig 6: Hero-Object Reachability
  ✓ 06_reachability.png
📊 Fig 7: Cluster Analysis
  ✓ 07_clusters.png
📊 Fig 8: Scoring & Route Simulation
  ✓ 08_scoring.png
📊 Fig 9: Object Proximity Network
   Graph: 100 nodes, 336 edges
  ✓ 09_network_graph.png
📊 Fig 10: Dendrogram
  ✓ 10_dendrogram.png
📊 Fig 11: Day-by-Day Progression
  ✓ 11_day_progression.png
📊 Fig 12: Correlation Matrix
  ✓ 12_correlations.png
📊 Fig 13: Optimal Hero Count Analysis
  ✓ 13_optimal_heroes.png
📊 Fig 14: Summary Table
  ✓ 14_summary_table.png

✅  All 14 figures saved!
📁  Output: /kaggle/wo